**Hypothesis: Do regional cost of living and labor market conditions predict discounting?**

This question is being asked in order to find significant links between economic conditions and in-store pricing. Using multiple regression, we would find statistical significance and practical significance.

**Y:** Total promotional markdown (discount) amount  
**X:** CPI (cost of living), Unemployment Rate

In [ ]:
%%sql -r dataframe_1
USE ROLE CHEETAH_ROLE;
USE DATABASE CHEETAH_DB;
USE SCHEMA SILVER_BUSINESS_DATA; 

In [ ]:
%%sql -r regression_data
SELECT STORE_ID, WEEK_DATE, CPI, UNEMPLOYMENT_RATE, FUEL_PRICE, STATE, STORE_TYPE, MARKDOWN1, MARKDOWN2, MARKDOWN3, MARKDOWN4, MARKDOWN5
FROM CHEETAH_DB.SILVER_BUSINESS_DATA.STORE_METRICS
WHERE MARKDOWN1 IS NOT NULL
   OR MARKDOWN2 IS NOT NULL
   OR MARKDOWN3 IS NOT NULL
   OR MARKDOWN4 IS NOT NULL
   OR MARKDOWN5 IS NOT NULL;

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

df = regression_data.to_pandas()
markdown_cols = ['MARKDOWN1', 'MARKDOWN2', 'MARKDOWN3', 'MARKDOWN4', 'MARKDOWN5']
df['TOTAL_MARKDOWN'] = df[markdown_cols].sum(axis=1)

# Drop rows where all markdowns are null or total is 0/NaN
df_clean = df.dropna(subset=['CPI', 'UNEMPLOYMENT_RATE', 'TOTAL_MARKDOWN'])
df_clean = df_clean[df_clean['TOTAL_MARKDOWN'] > 0].copy()

print(f"Records with valid markdowns and economic data: {len(df_clean)}")
print(f"\nDescriptive Statistics:")
print(df_clean[['CPI', 'UNEMPLOYMENT_RATE', 'FUEL_PRICE', 'TOTAL_MARKDOWN']].describe().round(2))

In [ ]:
X = df_clean[['CPI', 'UNEMPLOYMENT_RATE', 'FUEL_PRICE']]
y = df_clean['TOTAL_MARKDOWN']

X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())

In [ ]:

print(f"\nR-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
print(f"F-statistic p-value: {model.f_pvalue:.6f}")
print(f"\nThe model explains {model.rsquared*100:.1f}% of the variance in discounting.")
print(f"\nCoefficient Analysis (alpha = 0.05):")
for var in ['CPI', 'UNEMPLOYMENT_RATE', 'FUEL_PRICE']:
    coef = model.params[var]
    pval = model.pvalues[var]
    sig = "SIGNIFICANT" if pval < 0.05 else "NOT significant"
    print(f"  {var}:")
    print(f"    Coefficient = {coef:.4f}, p-value = {pval:.4f} -> {sig}")
    if pval < 0.05:
        direction = "increases" if coef > 0 else "decreases"
        print(f"    For each 1-unit increase in {var}, markdown {direction} by ${abs(coef):.2f}")

print(f"\nConclusion:")
if model.f_pvalue < 0.05:
    print("  The overall model IS statistically significant (p < 0.05).")
    print("  Regional economic conditions DO predict discounting behavior.")
else:
    print("  The overall model is NOT statistically significant (p >= 0.05).")
    print("  Regional economic conditions do NOT significantly predict discounting.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(10, 4))

predictors = ['CPI', 'UNEMPLOYMENT_RATE', 'FUEL_PRICE']
titles = ['CPI (Cost of Living)', 'Unemployment Rate', 'Fuel Price']

for i, (col, title) in enumerate(zip(predictors, titles)):
    axes[i].scatter(df_clean[col], df_clean['TOTAL_MARKDOWN'])
    z = np.polyfit(df_clean[col], df_clean['TOTAL_MARKDOWN'], 1)
    p = np.poly1d(z)
    axes[i].set_xlabel(title)
    axes[i].set_ylabel('Total Markdown ($)')
    axes[i].set_title(f'Markdown vs {title}')

plt.tight_layout()
plt.show()

# Interpretation of Statistical Analysis

The overall model is statistically significant meaning the relationship between economic predictors and markdown amounts is unlikely due to chance alone. All three predictors are individually significant at α = 0.05.

However, the model has low practical significance:
- R² = 0.014 — the model explains only **1.4%** of the variance in total markdown amounts
- Adjusted R² = 0.013 — confirms the poor explanatory power after adjusting for the number of predictors

This means that while CPI, unemployment rate, and fuel price have a statistically detectable relationship with discounting, they are **not meaningful predictors** in practice. Over 98% of the variation in markdown behavior is driven by other factors not included in this model (e.g., store strategy, inventory levels, seasonality, competition).

# Coefficient Interpretation

1. CPI  -> (Coefficient: -55.29 (Negative)) Higher cost of living is associated with *lower* markdowns ($55 less per 1-unit CPI increase)
2. Unemployment Rate -> (Coefficient: +618.92 (Positive)) Higher unemployment is associated with *higher* markdowns ($619 more per 1-percentage-point increase)
3. Fuel Price _> (Coefficient: -10,621.88 (Negative)) Higher fuel prices are associated with *lower* markdowns ($10,622 less per $1 increase in fuel)

# Conclusion
Cost of living, unemployment and gas prices play a role, but they play a very small one when it comes to store discounts. Economic conditions only account for 1% of variation in markdown spending. Discounts and markdown are driven by something else entirely, not by economic conditions of a stores locatin. 

In [ ]:
%%sql -r create_schema_result
CREATE SCHEMA IF NOT EXISTS CHEETAH_DB.GOLD_ANALYTICS;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

gold_df = df_clean[['STORE_ID', 'WEEK_DATE', 'STATE', 'STORE_TYPE', 'CPI', 'UNEMPLOYMENT_RATE', 'FUEL_PRICE', 'MARKDOWN1', 'MARKDOWN2', 'MARKDOWN3', 'MARKDOWN4', 'MARKDOWN5', 'TOTAL_MARKDOWN']].copy()
gold_df['PREDICTED_MARKDOWN'] = model.predict(X_const).round(2)
gold_df['RESIDUAL'] = (gold_df['TOTAL_MARKDOWN'] - gold_df['PREDICTED_MARKDOWN']).round(2)

snowpark_df = session.create_dataframe(gold_df)
snowpark_df.write.mode('overwrite').save_as_table('CHEETAH_DB.GOLD_ANALYTICS.STORE_MARKDOWN_REGRESSION')
session.table('CHEETAH_DB.GOLD_ANALYTICS.STORE_MARKDOWN_REGRESSION').show()